In [1]:
"Hello"

'Hello'

In [2]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline

## Loading Dataset

In [3]:
df:pd.DataFrame = pd.read_csv("./dataset/healthcare-dataset-stroke-data.csv")

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (5110, 12)
      id  gender   age  hypertension  heart_disease ever_married  \
0   9046    Male  67.0             0              1          Yes   
1  51676  Female  61.0             0              0          Yes   
2  31112    Male  80.0             0              1          Yes   
3  60182  Female  49.0             0              0          Yes   
4   1665  Female  79.0             1              0          Yes   

       work_type Residence_type  avg_glucose_level   bmi   smoking_status  \
0        Private          Urban             228.69  36.6  formerly smoked   
1  Self-employed          Rural             202.21   NaN     never smoked   
2        Private          Rural             105.92  32.5     never smoked   
3        Private          Urban             171.23  34.4           smokes   
4  Self-employed          Rural             174.12  24.0     never smoked   

   stroke  
0       1  
1       1  
2       1  
3       1  
4       1  


In [4]:
df.isnull().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

In [5]:
df.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   str    
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   str    
 6   work_type          5110 non-null   str    
 7   Residence_type     5110 non-null   str    
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   str    
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), str(5)
memory usage: 479.2 KB


# Basic Cleaning

In [7]:
df.drop("id", axis=1, inplace=True)

# Handle missing BMI values
df["bmi"] = df["bmi"].fillna(df["bmi"].median())

print("Missing values:\n", df.isnull().sum())

Missing values:
 gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64


# Encode Categorical Columns

In [8]:
categorical_cols = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

le_dict = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

# Split Features and Target

In [9]:
X = df.drop("stroke", axis=1)
y = df["stroke"]

print("Class Distribution:\n", y.value_counts())



Class Distribution:
 stroke
0    4861
1     249
Name: count, dtype: int64


# Handle Class Imbalance


In [10]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print("After SMOTE:\n", pd.Series(y_resampled).value_counts())

After SMOTE:
 stroke
1    4861
0    4861
Name: count, dtype: int64


# Train-Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled,
    y_resampled,
    test_size=0.2,
    random_state=42
)


# Feature Scaling

In [12]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# Train Models

In [13]:
from xgboost import train


models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

results = {}

print("Testing Models on Original Data:")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    training_accuracy = model.score(X_train, y_train)

    results[name] = {
        "model": model,
        "accuracy": accuracy,
        "auc": auc,
        "training accuracy": training_accuracy
    }

    print(f"\n{name}")
    print("Testing Accuracy:", accuracy)
    print("Training accuracy:", training_accuracy)
    print("AUC:", auc)
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Testing Models on Original Data:

Logistic Regression
Testing Accuracy: 0.8102827763496144
Training accuracy: 0.8127812781278128
AUC: 0.8916098334655036


              precision    recall  f1-score   support

           0       0.83      0.78      0.81       975
           1       0.79      0.84      0.81       970

    accuracy                           0.81      1945
   macro avg       0.81      0.81      0.81      1945
weighted avg       0.81      0.81      0.81      1945

Confusion Matrix:
 [[764 211]
 [158 812]]

Random Forest
Testing Accuracy: 0.9460154241645244
Training accuracy: 0.9998714157129999
AUC: 0.9894998678297647
              precision    recall  f1-score   support

           0       0.96      0.93      0.95       975
           1       0.93      0.97      0.95       970

    accuracy                           0.95      1945
   macro avg       0.95      0.95      0.95      1945
weighted avg       0.95      0.95      0.95      1945

Confusion Matrix:
 [[903  72]
 [ 33 937]]

XGBoost
Testing Accuracy: 0.9516709511568123
Training accuracy: 0.992027774205992
AUC: 0.9918033306899287
              precision    recall  f1-sco

d:\Mgit college\college projects\KRR\MedAi Expert\MedAiExpert\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:23:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


# Select Best Model

In [14]:

best_model_name = max(results, key=lambda x: results[x]["auc"])
best_model = results[best_model_name]["model"]

print("\nBest Model:", best_model_name)


Best Model: XGBoost


# Create Pipeline & Save All Models

In [15]:
pipelines = {}

for name in models.keys():
    model = results[name]["model"]

    pipeline = Pipeline([
        ("scaler", scaler),
        ("model", model)
    ])

    pipelines[name] = pipeline

    filename = f"models/{name.replace(' ', '_').lower()}_pipeline.pkl"
    joblib.dump(pipeline, filename)

    print(f"{name} pipeline saved as {filename}")

Logistic Regression pipeline saved as models/logistic_regression_pipeline.pkl
Random Forest pipeline saved as models/random_forest_pipeline.pkl
XGBoost pipeline saved as models/xgboost_pipeline.pkl


# Using the pickled model

In [17]:
import joblib
import pandas as pd

pipeline = joblib.load("models/xgboost_pipeline.pkl")

sample_input = pd.DataFrame([{
    "gender": 1,
    "age": 67,
    "hypertension": 0,
    "heart_disease": 1,
    "ever_married": 1,
    "work_type": 2,
    "Residence_type": 1,
    "avg_glucose_level": 228.69,
    "bmi": 36.6,
    "smoking_status": 1
}])

prediction = pipeline.predict(sample_input)
probability = pipeline.predict_proba(sample_input)

print("Prediction:", prediction)
print("Probability:", probability)


Prediction: [1]
Probability: [[0.145177 0.854823]]
